# Temperature softmax, entropy, greedy & inverse-CDF sampling

In [1]:
import numpy as np

logits = np.array([2.0, 1.0, 0.1, -1.0, 0.5])

def stable_softmax(logits, temperature=1.0):
    z = np.asarray(logits, dtype=float) / temperature
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()

def entropy(probs):
    p = np.asarray(probs, dtype=float)
    nz = p[p > 0]
    return float(-np.sum(nz * np.log(nz)))

def greedy_next_token(logits):
    return int(np.argmax(logits))

def sample_index(probs, u):
    c = np.cumsum(probs)
    i = int(np.searchsorted(c, u, side="left"))
    return min(i, len(probs) - 1)

def temperature_entropy(logits, temperatures):
    return np.array([entropy(stable_softmax(logits, T)) for T in temperatures])

In [2]:
p = stable_softmax(logits, 1.0)
print(np.round(p, 4))                                   # [0.5585 0.2055 0.0835 0.0278 0.1246]
print(round(entropy(p), 4))                             # 1.217
print(round(stable_softmax(logits, 2.0).max(), 4))      # 0.3719
print(round(stable_softmax(logits, 0.5).max(), 4))      # 0.8265
print(greedy_next_token(logits))                        # 0
print(sample_index(np.array([0.1,0.2,0.3,0.4]), 0.55))  # 2
print(stable_softmax(np.array([1000., 999., 5.])))      # no overflow
print(np.round(temperature_entropy(logits, [0.1, 0.5, 1, 2, 10, 100]), 4))

[0.5585 0.2055 0.0835 0.0278 0.1246]
1.217
0.3719
0.8265
0
2
[0.73105858 0.26894142 0.        ]
[5.0000e-04 6.2030e-01 1.2170e+00 1.4948e+00 1.6046e+00 1.6094e+00]
